In [2]:
import pandas as pd
import altair as alt

In [3]:
df = pd.read_csv('songs_cleaned.csv')
df = df.drop_duplicates()
df.columns = df.columns.str.strip()

df.head()

,Track,Album Name,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,...,TikTok Views,YouTube Playlist Reach,Apple Music Playlist Count,AirPlay Spins,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,Genre
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/24,1.0,725.4,3.904709e+08,30716.0,196631588.0,92.0,...,5.332282e+09,1.505970e+08,210.0,40975.0,114.0,18004655.0,22931.0,4818457.0,2669262.0,R&B/Soul
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/24,2.0,545.9,3.237039e+08,28113.0,174597137.0,92.0,...,2.083390e+08,1.563804e+08,188.0,40778.0,111.0,7780028.0,28444.0,6623075.0,1118279.0,Hip-Hop/Rap
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/24,3.0,538.4,6.013093e+08,54331.0,211607669.0,92.0,...,3.369121e+09,3.737850e+08,190.0,74333.0,172.0,5022621.0,5639.0,7208651.0,5285340.0,Indie Pop
3,Flowers,Flowers - Single,Miley Cyrus,1/12/23,4.0,444.9,2.031281e+09,269802.0,136569078.0,85.0,...,1.460373e+10,3.351189e+09,394.0,1474799.0,210.0,190260277.0,203384.0,NaN,11822942.0,Pop
4,Houdini,Houdini,Eminem,5/31/24,5.0,423.3,1.070349e+08,7223.0,151469874.0,88.0,...,NaN,1.127639e+08,182.0,12185.0,105.0,4493884.0,7006.0,207179.0,457017.0,Hip-Hop/Rap


In [25]:
df_bar = df[['Artist', 'Spotify Streams']].dropna()

top_artists = (
    df_bar.groupby('Artist', as_index=False)['Spotify Streams']
    .sum()
    .nlargest(25, 'Spotify Streams')
    .copy()
)
top_artists['Spotify Streams (B)'] = top_artists['Spotify Streams'] / 1000000000

chart2 = alt.Chart(top_artists).mark_bar(color='#1ED760').encode(
    x=alt.X('Spotify Streams (B):Q', title='Total Spotify Streams (Billions)'),
    y=alt.Y('Artist:N', sort='-x', title='Artist'),
    tooltip=[
        alt.Tooltip('Artist:N', title='Artist'),
        alt.Tooltip('Spotify Streams (B):Q', format='.2f', title='Streams (B)'),
    ]
).properties(
    title='Top 25 Artists by Total Spotify Streams (2024)',
    width=650,
    height=500
)

chart2

chart2.save('chart2.html')


__Takeaways:__
This bar chart ranks the top 25 artists by total Spotify streams across all of their songs found in the data set. Bad Bunny leads the list with over 37 billion streams, followed closely by The Weeknd at just under 37 billion streams and Drake just under 35 billion streams. Taylor Swift finishes the top four with over 34 billion streams. There is a noticeable drop off after the top 4 with Post Malone sitting at over 26 billion streams. After Ed Sheeran's 24 billion streams the rest have under 20 billion. This suggests that while many artists achieve incredible success on Spotify, there are only a few that dominate at the top. The top 4 have more than double the streams of the rest of the artists outside of the top 10. Each of the Top 4 is in a different genre, showing that the most streamed artists on Spotify don't have to be confined to a single genre. This chart is the total sum of streams across all songs in the data set for each artist. If an artist has a larger catalog of songs then they might rank higher despite having less incredibly popular songs.

__Design:__
A horizontal bar chart was chosen because it is the best way to compare a single quantitative variable across categorical items. The horizontal orientation was used to make the names of the artists also horizontal and easier to read. The bars are sorted in descending order from top to bottom allowing viewers to observe differences between each of the bars. Tooltips were added so viewers can hover over a bar and see the exact number of streams for each artist. A single color was used for all of the bars because there are no categories to distinguish.



In [27]:
df_scatter2 = df[['Track', 'Artist', 'Spotify Streams', 'Spotify Playlist Reach']].dropna().copy()
df_scatter2['Spotify Streams (B)'] = df_scatter2['Spotify Streams'] / 1000000000
df_scatter2['Spotify Playlist Reach (M)'] = df_scatter2['Spotify Playlist Reach'] / 1000000

slider = alt.binding_range(min=0, max=round(df_scatter2['Spotify Streams (B)'].max(), 2), step=0.01, name='Max Streams (B): ')
slider_param = alt.param(name='max_streams', bind=slider, value=df_scatter2['Spotify Streams (B)'].max())

# Top 50 artists ordered by total Spotify Streams
artist_order = (
    df_scatter2.groupby('Artist')['Spotify Streams']
    .sum()
    .sort_values(ascending=False)
    .head(50)
    .index.tolist()
)
artist_dropdown = alt.binding_select(options=[None] + artist_order, labels=['All Artists'] + artist_order, name='Artist: ')
artist_param = alt.param(name='artist_select', bind=artist_dropdown, value=None)

click = alt.selection_point(on='click', clear='dblclick')

chart3 = alt.Chart(df_scatter2).mark_circle(size=60, color='#1ED760').add_params(
    slider_param, artist_param, click
).transform_filter(
    'datum["Spotify Streams (B)"] <= max_streams'
).transform_filter(
    '!artist_select || datum.Artist === artist_select'
).encode(
    x=alt.X('Spotify Streams (B):Q', title='Spotify Streams (Billions)'),
    y=alt.Y('Spotify Playlist Reach (M):Q', title='Spotify Playlist Reach (Millions)'),
    opacity=alt.condition(click, alt.value(0.8), alt.value(0.06)),
    tooltip=[
        alt.Tooltip('Track:N', title='Track'),
        alt.Tooltip('Artist:N', title='Artist'),
        alt.Tooltip('Spotify Streams (B):Q', format='.3f', title='Streams (B)'),
        alt.Tooltip('Spotify Playlist Reach (M):Q', format='.2f', title='Playlist Reach (M)'),
    ]
).properties(
    title='Spotify Streams vs. Spotify Playlist Reach (2024)',
    width=650,
    height=450
)
chart3

chart3.save('chart3.html')

__Takeaways:__
This scatterplot examines the relationship between Spotify stream count and Spotify playlist reach for the songs in our data. We can observe a generally positive relationship between the two, suggesting that songs with higher streaming numbers tend to reach more playlists, meaning they reach broader audiences. The trend is not perfectly linear, with some variation being seen across the dataset, however, the overall pattern indicates that streaming popularity and playlist reach tend to move together. All of this implies that playlist placement may have an impact on a song's popularity and the streams it gets. With labels and artists being able to pay for playlisting, it might imply that more popular songs turn to these means to have their songs reach a wider range of listeners to further drive their streams. To view how individual artists are doing these things, viewers can also isolate any of the top 50 artists (by total streams) using the dropdown menu. This allows viewers to examine how a specific artist's catalog for the year is distributed across these metrics.

__Design:__
This scatterplot was designed to clearly communicate the relationship between playlisting and stream count on Spotify. A scatterplot was the most appropriate chart type for this data, given that we are examining the correlation between two continuous qualitative variables. Using a scatterplot makes it easier for viewers to observe the general distribution of the encoded points across the two axes. The measurements on these two axes were scaled to allow for easier readability. Three interactive features were added to this graph, with users having the ability to limit the number of songs shown by their streams, artists, and the option to click and single out a specific point. These decisions all allow for a balance between overview and detail in the visualization, making it useful as a tool for investigating individual artists and observations.

In [28]:
import numpy as np

df = pd.read_csv('songs_cleaned.csv')
df = df.drop_duplicates()
df.columns = df.columns.str.strip()

base_cols = ['Track', 'Artist', 'Genre', 'Spotify Streams',
             'TikTok Views', 'YouTube Views', 'Shazam Counts', 'TikTok Likes']

df_scatter = df[base_cols].copy()
df_scatter = df_scatter[df_scatter['Spotify Streams'] > 0]

#only keeps top 500 songs to remove clutter
df_scatter = df_scatter.nlargest(500, 'Spotify Streams')

df_scatter['Spotify Streams (M)'] = df_scatter['Spotify Streams'] / 1_000_000
df_scatter['TikTok Views (M)'] = df_scatter['TikTok Views'] / 1_000_000
df_scatter['YouTube Views (M)'] = df_scatter['YouTube Views'] / 1_000_000
df_scatter['Shazam Counts (K)'] = df_scatter['Shazam Counts'] / 1_000
df_scatter['TikTok Likes (M)'] = df_scatter['TikTok Likes'] / 1_000_000

brush = alt.selection_interval()

base_tooltip = [
    alt.Tooltip('Track:N', title='Track'),
    alt.Tooltip('Artist:N', title='Artist'),
    alt.Tooltip('Genre:N', title='Genre'),
    alt.Tooltip('Spotify Streams (M):Q', format='.1f', title='Spotify Streams (M)'),
]

def make_scatter(data, x_field, x_title, y_field, y_title, color_val, extra_tooltip, is_driver=False):
    x_min = data[x_field].min() * 0.95
    y_min = data[y_field].min() * 0.95

    points = alt.Chart(data).mark_circle(size=45).encode(
        x=alt.X(f'{x_field}:Q', title=x_title, scale=alt.Scale(domain=[x_min, data[x_field].max() * 1.05])),
        y=alt.Y(f'{y_field}:Q', title=y_title, scale=alt.Scale(domain=[y_min, data[y_field].max() * 1.05])),
        color=alt.condition(
            brush,
            alt.value(color_val),
            alt.value('#a0a0a0')
        ),
        opacity=alt.condition(
            brush,
            alt.value(0.7),
            alt.value(0.35)
        ),
        tooltip=base_tooltip + extra_tooltip
    )

    reg_line = points.transform_regression(
        x_field, y_field
    ).mark_line(
        color='black',
        strokeWidth=2,
        strokeDash=[6, 3]
    )

    chart = (points + reg_line).properties(
        width=320,
        height=280,
    )

    if is_driver:
        chart = chart.add_params(brush)
    return chart

df_tiktok = df_scatter.dropna(subset=['TikTok Views (M)', 'Spotify Streams (M)'])
df_youtube = df_scatter.dropna(subset=['YouTube Views (M)', 'Spotify Streams (M)'])
df_shazam = df_scatter.dropna(subset=['Shazam Counts (K)', 'Spotify Streams (M)'])
df_tiklikes = df_scatter.dropna(subset=['TikTok Likes (M)', 'Spotify Streams (M)'])

panel1 = make_scatter(
    df_tiktok,
    'TikTok Views (M)', 'TikTok Views (Millions)',
    'Spotify Streams (M)', 'Spotify Streams (Millions)',
    '#1DB954',
    [alt.Tooltip('TikTok Views (M):Q', format='.1f', title='TikTok Views (M)')],
    is_driver=True
).properties(title='Spotify Streams vs TikTok Views (drag to select)')

panel2 = make_scatter(
    df_youtube,
    'YouTube Views (M)', 'YouTube Views (Millions)',
    'Spotify Streams (M)', 'Spotify Streams (Millions)',
    '#FF0000',
    [alt.Tooltip('YouTube Views (M):Q', format='.1f', title='YouTube Views (M)')]
).properties(title='Spotify Streams vs YouTube Views')

panel3 = make_scatter(
    df_shazam,
    'Shazam Counts (K)', 'Shazam Counts (Thousands)',
    'Spotify Streams (M)', 'Spotify Streams (Millions)',
    '#0088FF',
    [alt.Tooltip('Shazam Counts (K):Q', format='.1f', title='Shazam Counts (K)')]
).properties(title='Spotify Streams vs Shazam Counts')

panel4 = make_scatter(
    df_tiklikes,
    'TikTok Likes (M)', 'TikTok Likes (Millions)',
    'Spotify Streams (M)', 'Spotify Streams (Millions)',
    '#FE2C55',
    [alt.Tooltip('TikTok Likes (M):Q', format='.1f', title='TikTok Likes (M)')]
).properties(title='Spotify Streams vs TikTok Likes')

multi_scatter = alt.vconcat(
    alt.hconcat(panel1, panel2),
    alt.hconcat(panel3, panel4)
).resolve_scale(
    color='independent'
).properties(
    title=alt.Title(
        'Top 500 Spotify Songs: Cross-Platform Performance',
        subtitle='Drag to select points on the TikTok Views panel to filter across all panels',
        fontSize=16,
        subtitleFontSize=12,
        anchor='start'
    )
).configure_view(
    strokeWidth=0
)

multi_scatter

multi_scatter.save('multi_scatter.html')

__Takeaways:__
These scatterplots examine the relationship between Spotify streams and TikTok views, YouTube views, Shazam count, and TikTok likes. In the Spotify Streams vs TikTok views scatterplot we see the regression line has a slight upward slope and that the points are very heavily clusters in the bottom left corner with a few scattered to the right. This shows that TikTok virality has a weak relationship with Spotify streaming success. In the Spotify Streams vs YouTube views scatterplot, the regression line has a steeper positive slope, and the points are spread more evenly along it. This is the strongest relationship out of the four plots implying that when songs do well on YouTube, they tend to do well on Spotify too. This makes sense because both are traditional streaming platforms where people can go to listen to music. TikTok is not a streaming platform where you can intentionally go to listen to music. The music is normally the background audio for many posts on the platform. The Spotify Streams vs Shazam counts has a moderate positive slope. This trend suggests that the songs people use shazam to find the name of also tend to have a lot of streams. This is likely because when someone hears a song and likes it, they want to find the name so they can keep listening to it after which would increase the spotify stream count. There is still a lot of scatter so having a high shazam count doesn't necessarily mean having high streams. Finally, the Spotify Streams vs TikTok likes plot has a very similar trend to TikTok views. The regression line positive but quite flat with a lot of heavy clustering on the bottom left. This reinforces that engagement on TikTok is largely independent of performance on Spotify. Overall, YouTube is the best predictor of Spotify success among these platforms, while TikTok metrics are poor predictors. This suggests that YouTube and Spotify have similar audiences where listeners are being intentional with what they are listening to.

__Design:__
This multifacted scatterplot was designed to allow viewers to compare how the top 500 songs on Spotify perform on other platforms at the same time. A scatterplot was used for each of the panels because we are examining the relationship between two quantitative variables. This makes it easier to observe the strength and direction of the correlation. A linear regression line was added to each panel to highlight the trend of each plot. This helps viewers understand which platforms have stronger relationships with Spotify streams. The data was filtered to the top 500 streamed songs on Spotify to focus the analysis on the most popular songs. The axes were scaled and adjusted to start at the minimum data values instead of 0 to maximize spaced used. The main interactive feature is a linked brush selection. When a viewer clicks and drags their mouse on the top left panel, it highlights the selected data on the other panels. The unselected data points turn grey to show they aren't selected. This cross filtering allows viewers to whether songs that cluster on one platform also cluster on the others.

In [29]:
import numpy as np

df = pd.read_csv('songs_cleaned.csv')
df = df.drop_duplicates()
df.columns = df.columns.str.strip()

platform_cols = [
    'Spotify Streams',
    'Spotify Playlist Count',
    'TikTok Views',
    'TikTok Likes',
    'YouTube Views',
    'YouTube Likes',
    'Shazam Counts'
]

df_corr = df[platform_cols].dropna()

corr_matrix = df_corr.corr(method='spearman')

corr_long = corr_matrix.reset_index().melt(id_vars='index')
corr_long.columns = ['Variable 1', 'Variable 2', 'Correlation']
corr_long['Correlation'] = corr_long['Correlation'].round(2)

var_order = platform_cols

heatmap = alt.Chart(corr_long).mark_rect(
    cornerRadius=4,
    stroke='white',
    strokeWidth=2
).encode(
    x=alt.X('Variable 1:N', title=None, sort=var_order,
             axis=alt.Axis(labelAngle=-30, labelFontSize=11, labelFontWeight='bold')),
    y=alt.Y('Variable 2:N', title=None, sort=var_order,
             axis=alt.Axis(labelFontSize=11, labelFontWeight='bold')),
    color=alt.Color('Correlation:Q',
                     scale=alt.Scale(scheme='redpurple', domain=[-1, 1]),
                     legend=alt.Legend(title='Correlation', titleFontSize=12, labelFontSize=11)),
    tooltip=[
        alt.Tooltip('Variable 1:N', title='Variable 1'),
        alt.Tooltip('Variable 2:N', title='Variable 2'),
        alt.Tooltip('Correlation:Q', title='Correlation', format='.2f')
    ]
).properties(
    width=550,
    height=550,
    title=alt.Title(
        'Cross-Platform Spearman Rank Correlation Matrix',
        subtitle='Rank correlation between streaming metrics across Spotify, TikTok, YouTube, and Shazam',
        fontSize=16,
        subtitleFontSize=12,
        anchor='start'
    )
)

text = alt.Chart(corr_long).mark_text(
    fontSize=13,
    fontWeight='bold'
).encode(
    x=alt.X('Variable 1:N', sort=var_order),
    y=alt.Y('Variable 2:N', sort=var_order),
    text=alt.Text('Correlation:Q', format='.2f'),
    color=alt.condition(
        'abs(datum.Correlation) > 0.7',
        alt.value('white'),
        alt.value('black')
    )
)

heatmap_chart = (heatmap + text).configure_view(
    strokeWidth=0
).configure_axis(
    grid=False
)

heatmap_chart

heatmap_chart.save('heatmap_chart.html')